In [ ]:
import os, time, json, urllib.request, urllib.error
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

ENV_FILE = find_dotenv(usecwd=True)          # sucht .env ab dem Notebook-Ordner (labs/.env)
print("Konfiguration aus:", ENV_FILE or "keine .env gefunden, Defaults (OpenAI, gpt-4.1)")
load_dotenv(ENV_FILE, override=True)   # override: CrewAI lädt beim Import schon eine .env aus einem Elternordner
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "https://api.openai.com/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4.1")

from pydantic import BaseModel
from crewai import Agent, Task, Crew, Process, LLM, TaskOutput

REPOS_FILE = "repos.txt"                                       # ein Repo pro Zeile (owner/name), # = Kommentar
FILE_PATH = "Test.java"                                        # Pfad zur Datei im Repo
BRANCH = "main"

zeilen = [z.strip() for z in Path(REPOS_FILE).read_text(encoding="utf-8").splitlines()]
REPOS = [z for z in zeilen if z and not z.startswith("#")]
print(f"{len(REPOS)} Repos aus {REPOS_FILE}:", REPOS)

# GitHub-API statt raw-URL: funktioniert auch für private Repos (Token aus GITHUB_TOKEN oder gh CLI)
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.popen("gh auth token 2>/dev/null").read().strip()

def load_code(repo):
    req = urllib.request.Request(
        f"https://api.github.com/repos/{repo}/contents/{FILE_PATH}?ref={BRANCH}",
        headers={"Accept": "application/vnd.github.raw", **({"Authorization": f"Bearer {GITHUB_TOKEN}"} if GITHUB_TOKEN else {})},
    )
    text = urllib.request.urlopen(req).read().decode("utf-8")
    return "\n".join(f"{i}: {z}" for i, z in enumerate(text.splitlines(), 1))   # Zeilennummern, LLMs zählen unzuverlässig

codes = {}                                   # repo -> Code mit Zeilennummern
for repo in REPOS:
    try:
        codes[repo] = load_code(repo)
        print(f"Datei geladen: {FILE_PATH} aus {repo}, {len(codes[repo].splitlines())} Zeilen")
    except urllib.error.HTTPError as e:      # Repo oder Datei fehlt: überspringen, Rest läuft weiter
        print(f"Übersprungen: {repo} ({e.code} {e.reason})")

conventions = Path("convetionals.md").read_text(encoding="utf-8").strip()
print("Konventionen:\n" + conventions)

llm = LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)
print("Modell:", llm.model, "| Endpunkt:", LLM_BASE_URL)
print("LLM antwortet:", llm.call("Antworte nur mit OK.")[:40])

In [14]:
conventions

'- Attribute mit Kleinbuchstaben\n- methoden mit Kleinbuchstaben\n- Klassennamen mit Großbuchstaben'

In [15]:
def make_llm(): 
    return LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)


In [16]:
# wir definieren zwei Agenten: Convention Checker und Technical Writer 
# role, goal, backstory => jeder Agent bekommt das 

def make_agents():
    analyst = Agent(
        role = "Convention Checker",
        goal = "Lese die Datei und prüfe jeden Klassennamen, jedes Attribut und jede Methode gegen die Namenskonventionen.",
        backstory = "Senior developer, welcher Code Reviews schreibt.",
        llm = make_llm(),
        max_iter = 3,
        verbose = True, 
    )

    writer = Agent(
        role = "Technical Writer",
        goal = "Das Prüfergebnis zusammenfassen für Entwickler.",
        backstory = "Kurz fassen, keine Marketing-Sprache, keine Spekulationen.",
        llm = make_llm(),
        max_iter = 3,
        verbose = True,
        )

    return analyst, writer

analyst, writer = make_agents()

In [ ]:
# zwei Tasks 

class Zusammenfassung(BaseModel):
    title: str
    attribute_ok: bool      # Attribute mit Kleinbuchstaben
    methoden_ok: bool       # Methoden mit Kleinbuchstaben
    klassen_ok: bool        # Klassennamen mit Großbuchstaben
    violations: list[str]

def make_tasks(analyst, writer, **task2_kwargs): 
    task1 = Task(
        description=(
            "Check the file {path} from the GitHub repository {repo}:\n\n{code}\n\n"
            "against these naming conventions:\n\n{conventions}\n\n"
            "The rules refer to the first letter of the name. Constructors are not methods. "
            "List every class name, attribute and method with its line number and whether it complies."
        ),
        expected_output=(
            "Three bullet lists titled 'Attributes', 'Methods' and 'Classes'. "
            "Each bullet names one identifier, its line number and OK or VIOLATION."
        ),
        agent=analyst,
    )
    task2 = Task(
        description=(
            "Write a check report from the analysis you received as context. "
            "Write title and violations in German. Each item in violations is "
            "one sentence of at most 120 characters and names identifier and line. "
            "Set attribute_ok, methoden_ok and klassen_ok to true only if the analysis "
            "reports no violation for that rule."
        ),
        expected_output=(
            "A Zusammenfassung: title (one line), attribute_ok (bool), methoden_ok (bool), "
            "klassen_ok (bool), violations (list of short sentences)."
        ),
        agent=writer,
        context=[task1],
        output_pydantic=Zusammenfassung,
        **task2_kwargs,
    )

    return task1, task2

task1, task2 = make_tasks(analyst, writer)

crew = Crew(agents=[analyst, writer], tasks=[task1,task2], process=Process.sequential, verbose=True)

# dieselbe Crew läuft einmal pro Repo, nacheinander (parallel ginge mit crew.kickoff_for_each_async)
summaries = {}                               # repo -> Zusammenfassung
for repo, code_text in codes.items():
    result = await crew.kickoff_async(inputs={"code": code_text, "conventions": conventions, "path": FILE_PATH, "repo": repo})
    summaries[repo] = result.pydantic

In [ ]:
for repo, s in summaries.items():
    print(f"{repo}: attribute_ok={s.attribute_ok}, methoden_ok={s.methoden_ok}, klassen_ok={s.klassen_ok}")
    for v in s.violations:
        print("  -", v)